# S07 — Tolan native-resolution GEDI-footprint aggregation

Tests the benchmark extraction operator for the very-high-resolution Tolan product and documents its effect on matched GEDI comparisons.

The public copy is output-stripped; authoritative exported tables and figures are distributed separately in the repository.

In [ ]:
from pathlib import Path
import json
import math
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import shapely
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform
from shapely.geometry import Point
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
PUBLICATION_ROOT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck")
OUT_DIR = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "Tolan_Native1m_Footprint_Audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RADIUS_M = 12.5
FOOTPRINT_AREA_M2 = math.pi * RADIUS_M**2
MIN_COVERAGE = 0.80
RECOMPUTE_TOLAN = False
USE_EARTH_ENGINE_IF_LOCAL_MISSING = True
EE_PROJECT = "gen-lang-client-0587204441"
TOLAN_ASSET = "projects/sat-io/open-datasets/facebook/meta-canopy-height"
TOLAN_BAND = "cover_code"

PRODUCT_ORDER = ["Our B4 Phase 2", "Pauls 2020", "Lang 2020", "Meta/Tolan 2023"]
DISPLAY_NAMES = {
    "Our B4 Phase 2": "Our Model",
    "Pauls 2020": "Pauls et al. 2024 (Pa24)",
    "Lang 2020": "Lang et al. 2023 (L23)",
    "Meta/Tolan 2023": "Tolan et al. 2024 (T24)",
}

SITES = {
    "Ifran": {
        "key": "Ifran_6", "eval_max": 45.0, "crs": "EPSG:32630",
        "catalog": PUBLICATION_ROOT / "Data/Dense/Ifran/Catalogs/final_catalog_C15_NATIVE",
        "coordinate_file": Path(r"E:\CHM\Ifran_6\DATA\GEDI\Output_Preproc_L2A\GEDI_QC_ACQ_clean.csv.gz"),
    },
    "Maamoura": {
        "key": "Maamoura", "eval_max": 20.0, "crs": "EPSG:32629",
        "catalog": PUBLICATION_ROOT / "Data/Low_Sparsity/Maamoura/Temporal_Catalogs/T4_DENSE_2019_2025_C15",
        "coordinate_file": None,
    },
}

LEGACY_CACHE = PROJECT / "Results/Final_Article_Harmonized_GEDIAnchored_NaturalP1/CHM_Comparison/GEDI_TEST_product_valid_support_signed_errors.csv.gz"
print("Output directory:", OUT_DIR)


In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()


def load_test_points(site, cfg):
    shots = pd.read_csv(cfg["catalog"] / "shot_catalog_step05.csv.gz", low_memory=False)
    shots = shots[shots["split"].astype(str).str.lower().eq("test")].copy()
    shots["rh95"] = pd.to_numeric(shots["rh95"], errors="coerce")
    shots = shots[shots["rh95"].between(2.0, cfg["eval_max"])].copy()
    shots["gedi_year"] = pd.to_datetime(shots["aux_gedi_date"], errors="coerce").dt.year

    if {"aux_lon", "aux_lat"}.issubset(shots.columns):
        shots["lon"] = pd.to_numeric(shots["aux_lon"], errors="coerce")
        shots["lat"] = pd.to_numeric(shots["aux_lat"], errors="coerce")
    else:
        coords = pd.read_csv(cfg["coordinate_file"], usecols=["shot_number", "lon", "lat"])
        coords["shot_number"] = pd.to_numeric(coords["shot_number"], errors="coerce").astype("Int64")
        shots["aux_shot_id"] = pd.to_numeric(shots["aux_shot_id"], errors="coerce").astype("Int64")
        shots = shots.merge(coords, left_on="aux_shot_id", right_on="shot_number", how="left", validate="many_to_one")

    shots = shots.dropna(subset=["aux_shot_uid", "lon", "lat", "rh95", "gedi_year"])
    shots = (shots.sort_values(["aux_shot_uid", "aux_abs_temporal_delta_days", "aux_gedi_date"], kind="stable")
                  .drop_duplicates("aux_shot_uid", keep="first"))
    shots["shot_id"] = shots["aux_shot_uid"].astype(str)
    return shots[["shot_id", "rh95", "gedi_year", "lon", "lat"]].reset_index(drop=True)


canonical_points = {site: load_test_points(site, cfg) for site, cfg in SITES.items()}
for site, frame in canonical_points.items():
    print(site, "canonical TEST shots:", len(frame), "unique:", frame.shot_id.nunique())
    assert len(frame) == frame.shot_id.nunique()


In [ ]:
def native_tolan_candidates(site, cfg):
    tag = cfg["crs"].replace(":", "")
    product = "Meta_WRI_Tolan_2023_CHM_native_1m"
    rel = Path(cfg["key"]) / product / "clean" / "mosaic" / f"{cfg['key']}__{product}__clean__{tag}.tif"
    return [
        PROJECT / "CHM_Products_Comparison" / rel,
        PUBLICATION_ROOT / "CHM_Products_Comparison" / rel,
    ]


def native_footprint_mean(source, x, y):
    footprint = Point(float(x), float(y)).buffer(RADIUS_M, quad_segs=32)
    raw = from_bounds(*footprint.bounds, transform=source.transform)
    col0 = max(0, int(math.floor(raw.col_off)) - 1)
    row0 = max(0, int(math.floor(raw.row_off)) - 1)
    col1 = min(source.width, int(math.ceil(raw.col_off + raw.width)) + 1)
    row1 = min(source.height, int(math.ceil(raw.row_off + raw.height)) + 1)
    if col1 <= col0 or row1 <= row0:
        return np.nan, 0.0
    window = Window(col0, row0, col1 - col0, row1 - row0)
    array = source.read(1, window=window, masked=True)
    values = np.ma.filled(array, np.nan).astype(float)
    masked = np.ma.getmaskarray(array)
    local_rows, local_cols = np.indices(values.shape)
    rows, cols = local_rows + row0, local_cols + col0
    affine = source.transform
    xa, xb = affine.c + cols * affine.a, affine.c + (cols + 1) * affine.a
    ya, yb = affine.f + rows * affine.e, affine.f + (rows + 1) * affine.e
    pixels = shapely.box(np.minimum(xa, xb), np.minimum(ya, yb), np.maximum(xa, xb), np.maximum(ya, yb))
    areas = shapely.area(shapely.intersection(pixels, footprint))
    valid = (areas > 1e-9) & (~masked) & np.isfinite(values) & (values >= 0) & (values <= 100)
    coverage = float(areas[valid].sum() / footprint.area)
    if coverage < MIN_COVERAGE or not valid.any():
        return np.nan, coverage
    return float(np.sum(values[valid] * areas[valid]) / areas[valid].sum()), coverage


def sample_local_native(points, path, target_crs):
    predictions = np.full(len(points), np.nan, dtype=np.float32)
    coverages = np.zeros(len(points), dtype=np.float32)
    with rasterio.open(path) as src:
        rx, ry = map(abs, src.res)
        if not (0.75 <= rx <= 1.25 and 0.75 <= ry <= 1.25):
            raise RuntimeError(f"Tolan raster is not native ~1 m: resolution={src.res}, path={path}")
        if str(src.crs) != target_crs:
            raise RuntimeError(f"Unexpected Tolan CRS {src.crs}; expected {target_crs}")
        xs, ys = transform("EPSG:4326", src.crs, points.lon.tolist(), points.lat.tolist())
        for i, (x, y) in enumerate(zip(xs, ys)):
            predictions[i], coverages[i] = native_footprint_mean(src, x, y)
    out = points.copy()
    out["prediction"] = predictions
    out["coverage"] = coverages
    out["aggregation_source"] = str(path)
    out["aggregation_method"] = "area-weighted mean of native ~1 m pixels within 12.5 m GEDI radius"
    return out[np.isfinite(out.prediction)].copy()


In [ ]:
def sample_gee_native(points, target_crs, batch_size=200):
    import ee
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)

    collection = ee.ImageCollection(TOLAN_ASSET)
    first = ee.Image(collection.first())
    bands = first.bandNames().getInfo()
    if TOLAN_BAND not in bands:
        raise RuntimeError(f"Expected Tolan band {TOLAN_BAND!r}; available bands={bands}")
    height = collection.mosaic().select(TOLAN_BAND).rename("height_m").toFloat()
    height = height.updateMask(height.gte(0).And(height.lte(100)))
    valid_area = ee.Image.pixelArea().updateMask(height.mask()).rename("valid_area_m2")

    rows = []
    for start in range(0, len(points), batch_size):
        chunk = points.iloc[start:start + batch_size]
        features = [
            ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]).buffer(RADIUS_M), {"shot_id": str(r.shot_id)})
            for r in chunk.itertuples(index=False)
        ]
        fc = ee.FeatureCollection(features)
        mean_info = height.reduceRegions(fc, ee.Reducer.mean(), scale=1, crs=target_crs, tileScale=4).getInfo()["features"]
        area_info = valid_area.reduceRegions(fc, ee.Reducer.sum(), scale=1, crs=target_crs, tileScale=4).getInfo()["features"]
        means = {f["properties"]["shot_id"]: f["properties"].get("mean") for f in mean_info}
        areas = {f["properties"]["shot_id"]: f["properties"].get("sum", 0.0) for f in area_info}
        for r in chunk.itertuples(index=False):
            sid = str(r.shot_id)
            pred = means.get(sid)
            coverage = float(areas.get(sid) or 0.0) / FOOTPRINT_AREA_M2
            rows.append({"shot_id": sid, "prediction": pred, "coverage": coverage})
        print(f"GEE native aggregation: {min(start + batch_size, len(points))}/{len(points)}", flush=True)

    sampled = points.merge(pd.DataFrame(rows), on="shot_id", how="left", validate="one_to_one")
    sampled["aggregation_source"] = TOLAN_ASSET
    sampled["aggregation_method"] = "Earth Engine native 1 m mean within 12.5 m GEDI radius"
    sampled = sampled[(sampled.coverage >= MIN_COVERAGE) & np.isfinite(sampled.prediction)].copy()
    return sampled


tolan_parts = []
provenance = {}
for site, cfg in SITES.items():
    cache_path = OUT_DIR / f"{site}_Tolan_native1m_GEDI_footprint_samples.csv.gz"
    native_path = next((p for p in native_tolan_candidates(site, cfg) if p.is_file()), None)
    if cache_path.is_file() and not RECOMPUTE_TOLAN:
        sampled = pd.read_csv(cache_path, low_memory=False)
        mode = "validated cache"
    elif native_path is not None:
        sampled = sample_local_native(canonical_points[site], native_path, cfg["crs"])
        mode = "local native raster"
        sampled.to_csv(cache_path, index=False, compression="gzip")
    elif USE_EARTH_ENGINE_IF_LOCAL_MISSING:
        sampled = sample_gee_native(canonical_points[site], cfg["crs"])
        mode = "Earth Engine native reduction"
        sampled.to_csv(cache_path, index=False, compression="gzip")
    else:
        raise FileNotFoundError(f"No native Tolan raster for {site}: {native_tolan_candidates(site, cfg)}")

    sampled["forest"] = site
    sampled["product"] = "Meta/Tolan 2023"
    sampled["error"] = sampled.prediction - sampled.rh95
    assert sampled.shot_id.is_unique
    assert (sampled.coverage >= MIN_COVERAGE).all()
    tolan_parts.append(sampled)
    provenance[site] = {"mode": mode, "n": len(sampled), "cache": str(cache_path), "native_path": str(native_path) if native_path else None}
    print(site, mode, "valid native-footprint samples:", len(sampled))

tolan_native = pd.concat(tolan_parts, ignore_index=True)
display(pd.DataFrame(provenance).T)


In [ ]:
legacy = pd.read_csv(LEGACY_CACHE, low_memory=False)
legacy = legacy[legacy.forest.isin(SITES) & legacy["product"].isin(PRODUCT_ORDER[:-1])].copy()
corrected = pd.concat([legacy, tolan_native], ignore_index=True, sort=False)

matched_parts = []
support_rows = []
for site in SITES:
    site_frame = corrected[corrected.forest.eq(site)].copy()
    id_sets = {
        product: set(site_frame.loc[site_frame["product"].eq(product), "shot_id"].astype(str))
        for product in PRODUCT_ORDER
    }
    common_ids = set.intersection(*id_sets.values())
    if len(common_ids) < 30:
        raise RuntimeError(f"{site}: insufficient corrected common support: n={len(common_ids)}")
    for product in PRODUCT_ORDER:
        part = site_frame[site_frame["product"].eq(product) & site_frame.shot_id.astype(str).isin(common_ids)].copy()
        if part.shot_id.astype(str).nunique() != len(common_ids):
            raise AssertionError(f"{site}/{product}: duplicate or missing common shots")
        matched_parts.append(part)
    support_rows.append({"forest": site, "common_n": len(common_ids), **{f"{p}_valid_n": len(ids) for p, ids in id_sets.items()}})

matched = pd.concat(matched_parts, ignore_index=True)
matched.to_csv(OUT_DIR / "matched_four_product_samples_Ifran_Maamoura_Tolan_native1m.csv.gz", index=False, compression="gzip")
support_audit = pd.DataFrame(support_rows)
support_audit.to_csv(OUT_DIR / "support_audit_Ifran_Maamoura.csv", index=False)
display(support_audit)


In [ ]:
def metrics(frame):
    y = frame.rh95.to_numpy(float)
    p = frame.prediction.to_numpy(float)
    return {
        "n": len(frame),
        "MAE_m": mean_absolute_error(y, p),
        "RMSE_m": mean_squared_error(y, p) ** 0.5,
        "R2": r2_score(y, p),
        "r": np.corrcoef(y, p)[0, 1],
        "bias_m": float(np.mean(p - y)),
        "slope": float(np.polyfit(y, p, 1)[0]),
        "SD_ratio": float(np.std(p, ddof=1) / np.std(y, ddof=1)),
    }


metric_rows = []
for (site, product), frame in matched.groupby(["forest", "product"], sort=False):
    metric_rows.append({"forest": site, "product": product, **metrics(frame)})
corrected_metrics = pd.DataFrame(metric_rows)
corrected_metrics.to_csv(OUT_DIR / "Table3_candidate_Ifran_Maamoura_Tolan_native1m.csv", index=False)
display(corrected_metrics)

# Explicitly compare old and corrected Tolan on the corrected common support.
old_tolan = pd.read_csv(LEGACY_CACHE, low_memory=False)
old_tolan = old_tolan[old_tolan.forest.isin(SITES) & old_tolan["product"].eq("Meta/Tolan 2023")].copy()
audit_rows = []
for site in SITES:
    ids = set(matched.loc[matched.forest.eq(site), "shot_id"].astype(str))
    old = old_tolan[old_tolan.forest.eq(site) & old_tolan.shot_id.astype(str).isin(ids)]
    new = matched[matched.forest.eq(site) & matched["product"].eq("Meta/Tolan 2023")]
    audit_rows.extend([
        {"forest": site, "Tolan_version": "legacy direct 10 m export", **metrics(old)},
        {"forest": site, "Tolan_version": "native ~1 m GEDI-footprint mean", **metrics(new)},
    ])
tolan_before_after = pd.DataFrame(audit_rows)
tolan_before_after.to_csv(OUT_DIR / "Tolan_legacy10m_vs_native1m_footprint_metrics.csv", index=False)
display(tolan_before_after)


In [ ]:
COLORS = {
    "Our B4 Phase 2": "#3B6FB6", "Pauls 2020": "#D49A00",
    "Lang 2020": "#2A9D8F", "Meta/Tolan 2023": "#B85C9E",
}

fig, axes = plt.subplots(2, 4, figsize=(13.2, 6.4), constrained_layout=True)
for row, (site, cfg) in enumerate(SITES.items()):
    axis_max = cfg["eval_max"]
    for col, product in enumerate(PRODUCT_ORDER):
        ax = axes[row, col]
        frame = matched[(matched.forest == site) & (matched["product"] == product)]
        h = ax.hexbin(frame.rh95, frame.prediction, gridsize=42, extent=(0, axis_max, 0, axis_max), mincnt=1, bins="log", cmap="viridis")
        ax.plot([0, axis_max], [0, axis_max], "--", color="0.25", lw=0.8)
        m = metrics(frame)
        ax.text(0.03, 0.97, f"R²={m['R2']:.2f}\nRMSE={m['RMSE_m']:.2f} m\nMAE={m['MAE_m']:.2f} m\nSlope={m['slope']:.2f}\nSD ratio={m['SD_ratio']:.2f}\nn={m['n']:,}", transform=ax.transAxes, va="top", fontsize=7.2, bbox=dict(facecolor="white", alpha=.88, edgecolor="0.7", pad=1.5))
        ax.set_xlim(0, axis_max); ax.set_ylim(0, axis_max); ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=.18)
        if row == 0: ax.set_title(DISPLAY_NAMES[product], fontsize=9.5, weight="bold")
        if col == 0: ax.set_ylabel(f"{site}\nCanopy-height estimate (m)")
        if row == 1: ax.set_xlabel("GEDI RH95 TEST (m)")

fig.savefig(OUT_DIR / "Fig8_candidate_Ifran_Maamoura_Tolan_native1m.pdf", dpi=300, bbox_inches="tight")
fig.savefig(OUT_DIR / "Fig8_candidate_Ifran_Maamoura_Tolan_native1m.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
manifest = {
    "status": "PASS",
    "sites": list(SITES),
    "excluded_site": "Agadir",
    "radius_m": RADIUS_M,
    "minimum_valid_coverage": MIN_COVERAGE,
    "tolan_asset": TOLAN_ASSET,
    "tolan_band": TOLAN_BAND,
    "legacy_tolan_10m_used": False,
    "aggregation": "native approximately 1 m values averaged within each 12.5 m radius GEDI footprint",
    "support": "intersection of identical canonical TEST shot identifiers across four products within each landscape",
    "provenance": provenance,
    "outputs": [str(p) for p in sorted(OUT_DIR.iterdir())],
}
(OUT_DIR / "audit_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("[PASS] No legacy Tolan 10 m raster was used.")
print("[PASS] Corrected Tolan values were aggregated at native resolution within GEDI footprints.")
print("[PASS] Ifran and Maamoura use strict four-product common shot support.")
